In [1]:
from typing import override
from torch import Tensor, nn
import torch
import numpy as np


# Network hyperparam
dim = 65
L = 4
mode = 8
width = 32
du = 3  # 2 for velociy x, y and pressure.

NU_LAYER = 3
da = du + 1

# NN
# Fourier Convolution Operator
class FCO(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()
        self.mode: int = mode
        
        self.R = nn.Parameter(torch.randn(mode, width, width, dtype=torch.complex128))

    @override
    def forward(self, x: Tensor) -> Tensor:
        # x of shape: (B, dim, dim, width)
        shape = x.shape
        x_hat = torch.fft.fftn(x, dim=(1, 2))[:, :self.mode, :self.mode, :]
        x_hat = torch.einsum('mij,bmni->bmnj', self.R, x_hat)
        x_hat = torch.fft.ifftn(x_hat, s=shape)
        x = x_hat.real
        return x


class PINOLayer(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()

        self.k: nn.Module = FCO(mode, width)
        self.w: nn.Module = nn.Linear(width, width, dtype=torch.float64)
        self.sigma: nn.Module = nn.GELU()

    @override
    def forward(self, x: Tensor) -> Tensor:
        w = self.w(x)
        k = self.k(x)
        x = self.sigma(w + k)
        return x


class PINO(nn.Module):
    def __init__(self, da: int, du: int, mode: int, width: int, L: int, T: int) -> None:
        super().__init__()
        self.T = T
        # uplift
        self.P = nn.Linear(da, width, dtype=torch.float64)
        # projection
        self.Q = nn.Linear(width, du, dtype=torch.float64)

        layers = [PINOLayer(mode, width) for _ in range(L)]
        self.core: nn.Module = nn.Sequential(
            self.P,
            *layers,
            self.Q,
        )

    def one_pass(self, x: Tensor) -> Tensor:
        return self.core(x)
    
    @override
    def forward(self, x: Tensor) -> Tensor:
        curr_x = x
        acc_list = []
        for i in range(self.T):
            next_x = self.one_pass(curr_x)
            acc_list.append(next_x.unsqueeze(1))  #  unsqueeze to add time dimension
            curr_x = torch.cat([next_x[..., :3], curr_x[..., 3:]], dim=-1)

        out = torch.cat(acc_list, dim=1)
        return out
            


# PDE param
T = (5, 10)
dimT = 50
alpha = 1
beta = 1

# re & nu not fixed

k = 1
dt = (T[1] - T[0]) / dimT
dx = 1.0 / dim

# Todo: fill in the choosen times
t = torch.arange(*T, dt)
x = torch.arange(0.0, 1.0, dx)
y = torch.arange(0.0, 1.0, dx)

# Training is done over elements of shape:
# batch, dimT, dim, dim, C
# We have to compute dimT elements before applying loss because pde loss needs it
# We could apply data loss more often but it would be a different scheme than L = Ldata + λLpde
# The batch dimension mixes data from different 

# To compute pde loss, we need pde params
# Dataset of shape a -> u
# Where a: (batch, dimT, dim, dim, C) with C = da (ex: du + broadcasted pde_params)
# u: (batch, dimT, dim, dim, C) with C = du

L2 = torch.nn.MSELoss(reduction="mean")


# a of shape (batch, dimT, dim, dim, da)
# PDE param embedded in da
# u  of shape (batch, dimT, dim, dim, du)

# residual loss over time
def residual_loss(a, u):
    # assumed density to be 1
    # v viscosity (nu)
    # du/dt = -(u.∇)u + v∇.∇u -∇p
    # R = du/dt + (u.∇)u + ∇p - v∇.∇u

    
    TRS = np.s_[:, 2:]  # Time restriction
    
    INT = np.s_[:, :, 1:-1, 1:-1, :]  # Interior
    XPO = np.s_[:, :,   2:, 1:-1, :]  # X Plus One
    XMO = np.s_[:, :,  :-2, 1:-1, :]  # X Minus One
    YPO = np.s_[:, :, 1:-1,   2:, :]  # Y Plus One
    YMO = np.s_[:, :, 1:-1,  :-2, :]  # Y Minus One 

    u_vel = u[..., :2]
    u_vel_t = u_vel[TRS]
    p_t = u[TRS][..., 2:2 + 1]
    
    nu = a[None][INT][..., NU_LAYER:NU_LAYER + 1]
    
    du_dt = (u_vel[INT][:, 2:] - u_vel[INT][:, 1:-1]) / dt
    
    du_dx = (u_vel_t[XPO] - u_vel_t[XMO]) / dx
    du_dy = (u_vel_t[YPO] - u_vel_t[YMO]) / dx
    du_dx2 = (u_vel_t[XPO] + u_vel_t[XMO] - 2 * u_vel_t[INT]) / (dx * dx)
    du_dy2 = (u_vel_t[YPO] + u_vel_t[YMO] - 2 * u_vel_t[INT]) / (dx * dx)
    dp_dx = (p_t[XPO] - p_t[XMO]) / dx
    dp_dy = (p_t[YPO] - p_t[YMO]) / dx
    
    c0 = u_vel_t[INT][..., 0] * du_dx[..., 0] + u_vel_t[INT][..., 1] * du_dy[..., 0]
    c1 = u_vel_t[INT][..., 0] * du_dx[..., 1] + u_vel_t[INT][..., 1] * du_dy[..., 1]
    convection = torch.stack([c0, c1], dim=-1)
    
    viscosity = nu * (du_dx2 + du_dy2)

    grad_p = torch.cat([dp_dx, dp_dy], dim=-1)
    
    R = du_dt + convection + grad_p - viscosity

    return torch.mean(R**2)

def batched_bcu(u):
    K = 1
    # top - slip
    u[..., 1:-1, 0, 0] = 2 * K - u[..., 1:-1, 1, 0]
    u[..., 1:-1, 0, 1] = - u[..., 1:-1, 1, 1]
    # bot - no slip
    u[..., 1:-1, -1, :] = - u[..., 1:-1, -2, :]
    # left - no slip
    u[..., 0, 1:-1, :] = - u[..., 1, 1:-1, :]
    # right - no slip
    u[..., -1, 1:-1, :] = - u[..., -2, 1:-1, :]


# boundary condition loss over time
def bc_loss(u):
    # compute BC
    
    u_vel = u[..., :2]
    bcu = u_vel.clone().detach()
    batched_bcu(bcu)
    return (
        L2(u_vel[:, :, 1:-1,    0, :], bcu[:, :, 1:-1,    0, :]) +
        L2(u_vel[:, :, 1:-1,   -1, :], bcu[:, :, 1:-1,   -1, :]) +
        L2(u_vel[:, :,    0, 1:-1, :], bcu[:, :,    0, 1:-1, :]) +
        L2(u_vel[:, :,   -1, 1:-1, :], bcu[:, :,   -1, 1:-1, :])
    )

# initial condition loss
def ic_loss(a, u):
    ic_vel = a[:, :, :, :2]
    u0_vel = u[:, 0, :, :, :2]
    return L2(ic_vel, u0_vel)

# L pde is J pde as long as loss is averaged over batch
def L_pde(a, pred):
    return residual_loss(a, pred) + alpha * bc_loss(pred) + beta * ic_loss(a, pred)


def full_loss(a, u, pred):
    return L_pde(a, pred) + L2(u, pred)

In [2]:
# pino = PINO(da, du, mode, width, L, dimT)

In [3]:
# Batch = 1
# test_input = torch.rand(Batch, dim, dim, da, dtype=torch.float64)

In [4]:
# test_input.shape

In [5]:
# output = pino(test_input)

In [6]:
# output.shape

In [7]:
# loss = L_pde(test_input, output)

In [8]:
# loss.backward()

In [9]:
# loss

In [10]:
import os
import zarr
import s3fs
from torch.utils.data import Dataset
import torch

with open(os.path.expanduser("~") + "/.keys") as f:
    for line in f:
        if line.startswith("export "):
            key, val = line.strip().split("=", 1)
            os.environ[key.replace("export ","")] = val


class LDCDataset(Dataset):
    def __init__(self):
        fs = s3fs.S3FileSystem(
            key=os.environ["AWS_ACCESS_KEY_ID"],
            secret=os.environ["AWS_SECRET_ACCESS_KEY"],
            client_kwargs={"endpoint_url": "http://localhost:8333"}
        )
        
        store_X = s3fs.S3Map(root="ldcdataset/X", s3=fs, check=False)
        store_Y = s3fs.S3Map(root="ldcdataset/Y", s3=fs, check=False)

        dts_X = zarr.open(store_X, mode="a")
        dts_Y = zarr.open(store_Y, mode="a")

        self.X = dts_X[:]
        self.Y = dts_Y[:]
        self.length = self.X.shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx])
        y = torch.tensor(self.Y[idx])
        return x, y

In [13]:
import torch
from torch import nn
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm


def train_epoch(model, loader):
    model.train()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        loss = criterion(x, y, model(x))
        
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4e}")
    return total / len(loader)

@torch.no_grad()
def test_epoch(model, loader):
    model.eval()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        
        loss = criterion(x, y, model(x))
        
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4e}")
    return total / len(loader)



lr = 5e-4
epochs = 5000

model = PINO(da, du, mode, width, L, dimT).cuda()
optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = full_loss


dataset = LDCDataset()
train_len = int(0.8 * len(dataset))
test_len = len(dataset) - train_len
train_dataset, test_dataset = random_split(dataset, [train_len, test_len])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)



for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    test_loss  = test_epoch(model, test_loader)
    sched.step()
    print(f"Epoch {epoch}: train_loss={train_loss:.4e}, test_loss={test_loss:.4e}")

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Epoch 0: train_loss=2.4754e+00, test_loss=2.2055e+00


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Epoch 119: train_loss=1.5817e+00, test_loss=1.5793e+00


  0%|          | 0/15 [00:00<?, ?it/s]

KeyboardInterrupt: 